In [0]:
# Create sample JSON files to simulate incoming data
import json

# Create a volume to store our files
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw_transactions")

# Write sample JSON files simulating hourly bank transactions
file1 = [
    {"id": 1, "customer": "Alice", "amount": 1500.00, "type": "credit", "date": "2026-05-19"},
    {"id": 2, "customer": "Bob",   "amount": 200.00,  "type": "debit",  "date": "2026-05-19"}
]

file2 = [
    {"id": 3, "customer": "Charlie", "amount": 5000.00, "type": "credit", "date": "2026-05-20"},
    {"id": 4, "customer": "Diana",   "amount": 450.00,  "type": "debit",  "date": "2026-05-20"}
]

# Write files to volume
dbutils.fs.put(
    "/Volumes/workspace/default/raw_transactions/batch1.json",
    json.dumps(file1),
    overwrite=True
)

dbutils.fs.put(
    "/Volumes/workspace/default/raw_transactions/batch2.json",
    json.dumps(file2),
    overwrite=True
)

print("✅ Sample files created!")
dbutils.fs.ls("/Volumes/workspace/default/raw_transactions/")

In [0]:
# Auto Loader — reads files incrementally
raw_stream = (spark.readStream
    .format("cloudFiles")                    # ← Auto Loader engine
    .option("cloudFiles.format", "json")     # ← files are JSON
    .option("cloudFiles.schemaLocation",     # ← tracks schema
        "/Volumes/workspace/default/raw_transactions/_schema")
    .load("/Volumes/workspace/default/raw_transactions/")  # ← source folder
)

# Check schema Auto Loader inferred
raw_stream.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define correct schema
schema = StructType([
    StructField("id",       IntegerType(), True),
    StructField("customer", StringType(),  True),
    StructField("amount",   DoubleType(),  True),
    StructField("type",     StringType(),  True),
    StructField("date",     StringType(),  True)
])

# Auto Loader with explicit schema
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation",
        "/Volumes/workspace/default/raw_transactions/_schema")
    .schema(schema)                          # ← explicit schema!
    .load("/Volumes/workspace/default/raw_transactions/")
)

raw_stream.printSchema()

In [0]:
# Write Auto Loader stream to Delta table
query = (raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation",
        "/Volumes/workspace/default/raw_transactions/_checkpoint")
    .option("mergeSchema", "true")           # ← handle schema evolution
    .trigger(availableNow=True)              # ← process all then stop
    .toTable("bronze_autoloader")            # ← target Delta table
)

query.awaitTermination()
print("✅ Auto Loader completed!")

# Check results
spark.sql("SELECT * FROM bronze_autoloader ORDER BY id").show()

In [0]:
# Simulate a new file arriving
file3 = [
    {"id": 5, "customer": "Eve",   "amount": 3000.00, "type": "credit", "date": "2026-05-21"},
    {"id": 6, "customer": "Frank", "amount": 750.00,  "type": "debit",  "date": "2026-05-21"}
]

dbutils.fs.put(
    "/Volumes/workspace/default/raw_transactions/batch3.json",
    json.dumps(file3),
    overwrite=True
)

print("✅ New file arrived!")

# Run Auto Loader again — should only process batch3!
query = (raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation",
        "/Volumes/workspace/default/raw_transactions/_checkpoint")
    .trigger(availableNow=True)
    .toTable("bronze_autoloader")
)

query.awaitTermination()
print("✅ Incremental load completed!")

# Check total records
spark.sql("SELECT COUNT(*) as total FROM bronze_autoloader").show()
spark.sql("SELECT * FROM bronze_autoloader ORDER BY id").show()

In [0]:
# List all files in the raw transactions folder
display(dbutils.fs.ls("/Volumes/workspace/default/raw_transactions/"))

# View schema folder
display(dbutils.fs.ls("/Volumes/workspace/default/raw_transactions/_schema/"))

# First check what's inside the schema folder
schema_content = display(dbutils.fs.ls("/Volumes/workspace/default/raw_transactions/_schema/_schemas/0"))

print(schema_content)

In [0]:
schema_content = dbutils.fs.head(
    "/Volumes/workspace/default/raw_transactions/_schema/_schemas/0"
)
print(schema_content)

# See how Auto Loader tracks processed files
display(dbutils.fs.ls(
    "/Volumes/workspace/default/raw_transactions/_checkpoint/"
))

In [0]:
# Check sources — processed files
display(dbutils.fs.ls(
    "/Volumes/workspace/default/raw_transactions/_checkpoint/sources/"
))

# Check inside sources/0/
display(dbutils.fs.ls(
    "/Volumes/workspace/default/raw_transactions/_checkpoint/sources/0/"
))

# Read sources metadata — shows tracked files!
content = dbutils.fs.head(
    "/Volumes/workspace/default/raw_transactions/_checkpoint/sources/0/metadata"
)
print(content)

# Check rocksdb folder
display(dbutils.fs.ls(
    "/Volumes/workspace/default/raw_transactions/_checkpoint/sources/0/rocksdb/"
))

